# Notebook 02 - Model B: property + OSM/GIS location predictors
Model B adds the GIS-derived distances (QGIS, OpenStreetMap) to the Model A property variables. It uses the same
target, the same cleaned records, **the same train/test split** (loaded from `split_ids.csv`), the same CV folds and
the same metrics. The difference from Model A shows how much of the location effect the OSM variables capture.

This notebook also contains the **GIS variable results** (Chapter 4.3).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy import stats

from src import config as C, data as D, modelling as M, runner as R, plots as P, interpret as I
P.setup_style()
pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", lambda v: f"{v:,.3f}")
T, F = C.TABLES_DIR, C.FIGURES_DIR
def show(fig, name):
    P.save(fig, name); plt.show()

In [2]:
data = D.load_and_prepare(verbose=False)
lab, train, test = data["labelled"], data["train"], data["test"]
print(len(lab), "records;", len(train), "train /", len(test), "test (same split as Model A)")

502 records; 401 train / 101 test (same split as Model A)


## 1. GIS-derived distances (Chapter 4.3)

In [3]:
rows = []
for g in C.GIS_FEATURES:
    s = data["all"][g]
    rows.append({"Variable": C.label(g), "Group": C.GIS_GROUPS[g], "In Table y (3.5)": g in C.GIS_IN_METHODOLOGY,
                 "Records": s.notna().sum(), "Missing": s.isna().sum(), "Min": s.min(), "Median": s.median(),
                 "Max": s.max(), "Range": s.max() - s.min()})
gis_desc = pd.DataFrame(rows)
gis_desc.to_csv(T / "B_gis_descriptive.csv", index=False)
gis_desc

,Variable,Group,In Table y (3.5),Records,Missing,Min,Median,Max,Range
0,Distance to A/B road (m),Road accessibility,True,566,0,61.807,885.432,"1,661.934","1,600.127"
1,Distance to bus stop (m),Public transport,True,566,0,7.045,206.333,988.903,981.858
2,Distance to railway station (m),Public transport,True,566,0,165.502,589.685,"1,033.815",868.313
3,Distance to CBD (m),Centrality,True,566,0,202.347,889.204,"1,341.693","1,139.346"
4,Distance to school (m),Neighbourhood amenities,True,566,0,24.672,341.078,874.753,850.081
5,Distance to hospital (m),Neighbourhood amenities,True,566,0,"1,040.288","1,517.675","1,730.546",690.257
6,Distance to supermarket (m),Neighbourhood amenities,True,566,0,186.301,448.323,855.437,669.136
7,Distance to bank (m),Neighbourhood amenities,False,566,0,59.883,327.984,577.338,517.455
8,Distance to pharmacy (m),Neighbourhood amenities,False,566,0,14.437,305.956,616.920,602.483
9,Distance to playground (m),Neighbourhood amenities,False,566,0,"1,079.549","1,324.871","2,079.208",999.659


In [4]:
show(P.gis_distributions(lab), "4_3_gis_distributions.png")

All 11 distances are available for every property, so no OSM gaps had to be treated as missing.
The distances span a few hundred metres to about 2 km within the ward.

### Association with Annual Value and with Annual Value per sq ft

In [5]:
def gis_corr(target):
    out = []
    for g in C.GIS_FEATURES:
        rho, p = stats.spearmanr(lab[g], lab[target])
        out.append({"Variable": g, "Label": C.label(g), "Spearman rho": rho, "Spearman p-value": p})
    return pd.DataFrame(out)
tc_av, tc_rate = gis_corr(C.TARGET), gis_corr("AV_per_sqft")
gis_corr_table = tc_av.merge(tc_rate, on=["Variable", "Label"], suffixes=(" (AV)", " (AV per sq ft)"))
gis_corr_table.to_csv(T / "B_gis_correlations.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
P.gis_target_correlation(tc_av, ax[0], "(a) with Annual Value")
P.gis_target_correlation(tc_rate, ax[1], "(b) with Annual Value per sq ft")
ax[1].set_yticklabels([])
ax[1].set_yticks(range(len(tc_rate)))
ax[1].set_yticklabels([C.SHORT_LABELS[v] for v in tc_rate.sort_values("Spearman rho")["Variable"]])
fig.tight_layout()
show(fig, "4_4_gis_correlations.png")
gis_corr_table

,Variable,Label,Spearman rho (AV),Spearman p-value (AV),Spearman rho (AV per sq ft),Spearman p-value (AV per sq ft)
0,Dist_AB_Road,Distance to A/B road (m),-0.058,0.191,0.314,0.000
1,Dist_Bus_Stop,Distance to bus stop (m),-0.120,0.007,0.169,0.000
2,Dist_Railway,Distance to railway station (m),0.219,0.000,-0.264,0.000
3,Dist_CBD,Distance to CBD (m),0.175,0.000,-0.270,0.000
4,Dist_School,Distance to school (m),-0.137,0.002,0.220,0.000
5,Dist_Hospital,Distance to hospital (m),-0.015,0.738,-0.253,0.000
6,Dist_Supermarket,Distance to supermarket (m),-0.079,0.078,0.280,0.000
7,Dist_Bank,Distance to bank (m),-0.021,0.634,0.271,0.000
8,Dist_Pharmacy,Distance to pharmacy (m),0.020,0.649,0.232,0.000
9,Dist_Playground,Distance to playground (m),-0.032,0.476,0.251,0.000


**Why AV per sq ft?** Annual Value mostly scales with floor area. Location should therefore appear in the
**rate per sq ft** that CMC applies through the road-wise chart. Panel (b) shows whether the distances relate to that
rate.

### Multicollinearity among the distances

In [6]:
corr = lab[C.GIS_FEATURES].corr()
corr.to_csv(T / "B_gis_corr_matrix.csv")
show(P.correlation_heatmap(corr, "Pearson correlation between GIS distances"), "4_5_gis_correlation_matrix.png")
vif_all = M.vif_table(train, ["Age", "Log_Total_Floor_Area", "Num_Floors"] + C.GIS_FEATURES)
vif_all.to_csv(T / "B_vif_all_gis.csv", index=False)
vif_all

,Variable,VIF,VIF > 5
0,Dist_CBD,"3,906.404",True
1,Dist_Railway,"3,228.632",True
2,Dist_Shopping_Mall,"1,166.401",True
3,Dist_Playground,722.885,True
4,Dist_Supermarket,471.841,True
5,Dist_Hospital,201.136,True
6,Dist_School,179.444,True
7,Dist_Bus_Stop,47.370,True
8,Dist_Pharmacy,34.182,True
9,Dist_Bank,31.388,True


Within one ward, the distances to the CBD, railway station, schools, supermarkets and so on largely encode the
**same position**, so many pairs are highly correlated and the VIFs are very large. A linear regression with all 11
would give unstable, uninterpretable coefficients. MLR therefore uses a **screened subset** (rule below, applied to
the training rows only). RF and XGBoost are not affected in the same way and use all 11 distances.

In [7]:
sets, screened, screen_table = M.feature_sets(train)
screen_table.to_csv(T / "B_mlr_gis_screening.csv", index=False)
vif_screened = M.vif_table(train, ["Age", "Log_Total_Floor_Area", "Num_Floors"] + screened)
vif_screened.to_csv(T / "B_vif_screened.csv", index=False)
display(screen_table)
print("GIS variables kept for MLR:", screened)
vif_screened

,Variable,Group,|r| with log(Annual Value),Decision,Reason
0,Dist_Railway,Public transport,0.197,Retained,No |r| >= 0.70 with a retained variable; VIF <= 5
1,Dist_CBD,Centrality,0.173,Dropped,|r| = 0.98 with retained Dist_Railway
2,Dist_School,Neighbourhood amenities,0.142,Dropped,|r| = 0.95 with retained Dist_Railway
3,Dist_Supermarket,Neighbourhood amenities,0.115,Dropped,|r| = 0.89 with retained Dist_Railway
4,Dist_Bus_Stop,Public transport,0.092,Dropped,|r| = 0.76 with retained Dist_Railway
5,Dist_AB_Road,Road accessibility,0.072,Retained,No |r| >= 0.70 with a retained variable; VIF <= 5
6,Dist_Bank,Neighbourhood amenities,0.067,Dropped,|r| = 0.71 with retained Dist_Railway
7,Dist_Shopping_Mall,Neighbourhood amenities,0.063,Dropped,|r| = 0.85 with retained Dist_Railway
8,Dist_Playground,Neighbourhood amenities,0.035,Dropped,|r| = 0.78 with retained Dist_Railway
9,Dist_Hospital,Neighbourhood amenities,0.032,Retained,No |r| >= 0.70 with a retained variable; VIF <= 5


GIS variables kept for MLR: ['Dist_AB_Road', 'Dist_Railway', 'Dist_Hospital']


,Variable,VIF,VIF > 5
0,Dist_Railway,2.379,False
1,Condition_Good,2.219,False
2,Log_Total_Floor_Area,2.139,False
3,Dist_Hospital,2.043,False
4,Dist_AB_Road,1.934,False
5,Condition_Fair+,1.797,False
6,Age,1.514,False
7,Num_Floors,1.487,False
8,Condition_Fair-,1.246,False
9,Condition_Poor,1.103,False


## 2. Model B predictors

In [8]:
categories = M.condition_categories(lab["Condition"])
pd.DataFrame([{"Algorithm": C.MODEL_NAMES[k], "n predictors": len(sets[("B", k)]),
               "Model B predictors": ", ".join(sets[("B", k)])} for k in C.ALGORITHMS])

,Algorithm,n predictors,Model B predictors
0,Multiple Linear Regression,7,"Age, Condition, Log_Total_Floor_Area, Num_Floo..."
1,Random Forest,15,"Age, Condition, Total_Floor_Area, Num_Floors, ..."
2,XGBoost,15,"Age, Condition, Total_Floor_Area, Num_Floors, ..."


## 3. Train, tune and validate Model B

In [9]:
outB = R.run_set("B", data, sets, categories, permutation=True)
perfB, tuneB, ratioB = R.summary_tables(outB)
cols = ["Algorithm", "CV RMSE", "CV RMSE SD", "CV MAE", "CV R2 pooled", "Train R2", "Test RMSE", "Test MAE", "Test R2"]
perfB[cols]

Model B - property + GIS | Multiple Linear Regression  CV RMSE   13,119 CV R2 pooled 0.641 | test RMSE    6,346 R2 0.930 | 8s


Model B - property + GIS | Random Forest               CV RMSE   11,468 CV R2 pooled 0.803 | test RMSE    5,973 R2 0.938 | 44s


Model B - property + GIS | XGBoost                     CV RMSE   11,248 CV R2 pooled 0.810 | test RMSE    5,768 R2 0.942 | 18s


,Algorithm,CV RMSE,CV RMSE SD,CV MAE,CV R2 pooled,Train R2,Test RMSE,Test MAE,Test R2
0,Multiple Linear Regression,"13,119.307","11,917.382","4,086.432",0.641,0.708,"6,345.987","3,451.609",0.930
1,Random Forest,"11,468.491","6,327.478","3,808.183",0.803,0.921,"5,973.176","2,804.682",0.938
2,XGBoost,"11,247.784","6,249.478","3,739.854",0.810,0.965,"5,767.699","2,932.160",0.942


In [10]:
tuneB

,Model set,Algorithm,Predictors,Predictor list,Tuned hyperparameters
0,Model B - property + GIS,Multiple Linear Regression,7,"Age, Condition, Log_Total_Floor_Area, Num_Floo...",none (OLS)
1,Model B - property + GIS,Random Forest,15,"Age, Condition, Total_Floor_Area, Num_Floors, ...","{""max_depth"": 15, ""max_features"": 1.0, ""min_sa..."
2,Model B - property + GIS,XGBoost,15,"Age, Condition, Total_Floor_Area, Num_Floors, ...","{""subsample"": 0.6, ""reg_lambda"": 1, ""reg_alpha..."


In [11]:
show(P.actual_vs_predicted(test[C.TARGET], {k: r["pred_test"] for k, r in outB["results"].items()},
                           C.MODEL_SETS["B"]), "B_actual_vs_predicted.png")

## 4. MLR coefficients (Model B, screened GIS set)

In [12]:
olsB, olsB_table, olsB_stats = M.ols_fit(train, sets[("B", "mlr")], categories)
olsB_table.to_csv(T / "B_mlr_ols.csv", index=False)
pd.DataFrame([olsB_stats]).to_csv(T / "B_mlr_ols_fit.csv", index=False)
display(pd.DataFrame([olsB_stats]))
olsB_table[["Term", "Coefficient (log scale)", "Std. error", "p-value", "Readable effect"]]

,n,R2 (log scale),Adj. R2 (log scale),F p-value,AIC,Condition number
0,401,0.935,0.934,0.000,-62.192,"38,028.009"


,Term,Coefficient (log scale),Std. error,p-value,Readable effect
0,const,3.399,0.219,0.000,
1,Age,-0.000,0.001,0.687,-0.48 % per +10 years
2,Log_Total_Floor_Area,0.976,0.020,0.000,+9.75 % per +10 % floor area (elasticity 0.976)
3,Num_Floors,-0.088,0.019,0.000,-8.41 % per extra floor (same total area)
4,Dist_AB_Road,0.000,0.000,0.003,+1.24 % per +100 m
5,Dist_Railway,-0.000,0.000,0.875,-0.12 % per +100 m
6,Dist_Hospital,-0.000,0.000,0.787,-0.30 % per +100 m
7,Condition_Poor,-0.352,0.117,0.003,-29.70 % vs Fair
8,Condition_Fair-,-0.225,0.063,0.000,-20.11 % vs Fair
9,Condition_Fair+,0.093,0.030,0.002,+9.80 % vs Fair


## 5. Assessment-ratio analysis for Model B

In [13]:
ratioB[["Algorithm", "Sample", "n", "Median ratio", "COD (%)", "PRD", "PRB", "Within +/-10% (%)", "IAAO check"]]

,Algorithm,Sample,n,Median ratio,COD (%),PRD,PRB,Within +/-10% (%),IAAO check
0,Multiple Linear Regression,Hold-out test,101,1.020,22.046,1.166,-0.134,55.446,level OK; COD > 15; PRD regressive
1,Multiple Linear Regression,Training (out-of-fold CV),401,1.014,13.717,1.044,-0.000,59.850,level OK; uniformity OK; PRD regressive
2,Random Forest,Hold-out test,101,1.005,19.933,1.140,-0.118,69.307,level OK; COD > 15; PRD regressive
3,Random Forest,Training (out-of-fold CV),401,0.993,11.682,1.045,-0.012,63.092,level OK; uniformity OK; PRD regressive
4,XGBoost,Hold-out test,101,0.998,20.328,1.141,-0.112,65.347,level OK; COD > 15; PRD regressive
5,XGBoost,Training (out-of-fold CV),401,0.988,11.619,1.052,-0.018,64.589,level OK; uniformity OK; PRD regressive


In [14]:
outA_test = set(pd.read_csv(C.SPLIT_FILE).query("Partition == 'test'")["Property_ID"])
assert set(outB["test_ids"]) == outA_test, "Model B must be tested on the same records as Model A"
print("Model B complete - outputs saved with prefix B_ in outputs/")

Model B complete - outputs saved with prefix B_ in outputs/
